In [ ]:
import pathlib

# import pandas as pd
import jax.random as jr
from torch.utils.data import DataLoader
# import altair as alt

import trapallada.data
import trapallada.plot

So that imported modules are automatically reloaded

In [ ]:
%load_ext autoreload
%autoreload 2

# Images

The *base* file name for the data

In [ ]:
# base_filename = pathlib.Path('small.pt')
base_filename = pathlib.Path.home() / 'Sync' / 'research' / 'light curves' / 'small.pt'

`Dataset` is loaded: 18,000 gray-scale (i.e., single-channel) images

In [ ]:
ds = trapallada.data.ImageDataset(
    base_filename.with_stem(base_filename.stem + '_imgs'),
    base_filename.with_stem(base_filename.stem + '_labels').with_suffix('.parquet'),
)
ds.imgs.shape

There is an internal `pd.DataFrame` storing all the metadata

In [ ]:
ds.df

Using a `DataLoader`

In [ ]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [ ]:
batch = next(iter(dl))
len(batch)

- Indexes

- Images

- Labels

In [ ]:
batch[0], batch[1].shape, batch[2]

# Signals

Input directory

In [ ]:

input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

Individual files with

- the actual signals, and

- a `DataFrame` with metadata.

In [ ]:
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

A *PyTorch* `Dataset`

In [ ]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

It encompasses a *PyTorch* `Tensor` for the signals

In [ ]:
ds.signals.shape

and a *Pandas* `DataFrame`

In [ ]:
ds.metadata_df.sample(n=10)

In [ ]:
mean = ds.signals.mean(axis=-1)
std = ds.signals.std(axis=-1)
mean, std

One can also *normalize* the `DataSet`

In [ ]:
normalized_ds = trapallada.data.SignalsDataset(signals_file, metadata_file,normalize=True)
mean = normalized_ds.signals.mean(axis=-1)
std = normalized_ds.signals.std(axis=-1)
mean, std

Using a `DataLoader`

In [ ]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [ ]:
batch = next(iter(dl))
len(batch)

- Indexes

- Images

- Labels

In [ ]:
batch[0], batch[1].shape, batch[2]

Let us plot the first

In [ ]:
n_samples_to_plot = 1_000

samples of the first signal in the batch

In [ ]:
trapallada.plot.signal(batch[1][0][0, 0:n_samples_to_plot].numpy())

Some random signals

In [ ]:
n_signals = 15
i_signals = jr.randint(jr.PRNGKey(42), (n_signals,), 0, len(ds)).tolist()
signals = ds.signals[i_signals]
signals_df = ds.metadata_df.iloc[i_signals]
signals_df

In [ ]:
trapallada.plot.signals_dataset(ds, n_samples=1_000)